In [ ]:
harmony_host_url = "https://harmony.uat.earthdata.nasa.gov"

In [ ]:
from pathlib import Path
from tempfile import TemporaryDirectory

import harmony

environments = {
    "http://localhost:3000": harmony.Environment.LOCAL,
    "https://harmony.sit.earthdata.nasa.gov": harmony.Environment.SIT,
    "https://harmony.uat.earthdata.nasa.gov": harmony.Environment.UAT,
    "https://harmony.earthdata.nasa.gov": harmony.Environment.PROD,
}
assert harmony_host_url in environments

harmony_client = harmony.Client(env=environments[harmony_host_url])
request = harmony.Request(
    collection=harmony.Collection(id="NISAR_L2_GCOV_BETA_V1"),
    granule_name=[
        "NISAR_L2_PR_GCOV_015_156_A_010_2005_DVDV_A_20230619T000803_20230619T000818_T05000_N_P_J_001"
    ],
    format="image/png",
)
job_id = harmony_client.submit(request)
harmony_client.wait_for_processing(job_id)

with TemporaryDirectory() as temp_dir:
    urls = list(harmony_client.result_urls(job_id))

    assert len(urls) == 109
    assert len([url for url in urls if url.endswith(".txt")]) == 1
    assert len([url for url in urls if url.endswith(".png")]) == 36
    assert len([url for url in urls if url.endswith(".pgw")]) == 36
    assert len([url for url in urls if url.endswith(".png.aux.xml")]) == 36

    output_files = [
        Path(harmony_client.download(url, temp_dir).result()) for url in urls[1:4]
    ]

    reference_data = Path("reference_data")
    assert output_files[0].read_bytes() == (reference_data / "rgb.png").read_bytes()
    assert output_files[1].read_bytes() == (reference_data / "rgb.pgw").read_bytes()
    assert (
        output_files[2].read_bytes()
        == (reference_data / "rgb.png.aux.xml").read_bytes()
    )